# QA RAG Pipeline

In [ ]:
! pip install qdrant-client

In [ ]:
import os
import pandas as pd
from openai import OpenAI
from datasets import load_dataset
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from qdrant_client.models import VectorParams, Distance, PointStruct

In [ ]:
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL"), 
    api_key=os.getenv("QDRANT_API_KEY")
)

collection_name = "mental_health_kb"

collections = qdrant_client.get_collections()
collection_names = [c.name for c in collections.collections]

if collection_name in collection_names:
    qdrant_client.delete_collection(collection_name)
    print("Collection deleted.")
else:
    print("Collection does not exist.")

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)  # 384 = dim for all-MiniLM-L6-v2
)

Collection deleted.


True

In [10]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

ds = load_dataset("Amod/mental_health_counseling_conversations")

points = []
for i, row in enumerate(ds["train"]):
    vector = embedder.encode(row["Context"]).tolist()
    points.append(PointStruct(
        id=i,
        vector=vector,
        payload={"context": row["Context"], "response": row["Response"]}
    ))

batch_size = 100
for i in range(0, len(points), batch_size):
    qdrant_client.upsert(collection_name=collection_name, points=points[i:i+batch_size])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
gpt_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

def retrieve_context(query, top_k=3):
    query_vector = embedder.encode(query).tolist()
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k
    )
    return [
        {"context": hit.payload["context"], "response": hit.payload["response"], "score": hit.score} for hit in results.points
    ]

def generate_answer(user_query, emotion=None):
    retrieved = retrieve_context(user_query)
    
    context_block = "\n\n".join(
        f"Similar past conversation:\nQ: {r['context']}\nA: {r['response']}" for r in retrieved
    )
    
    emotion_note = f"\nThe user seems to be feeling: {emotion}. Adjust your tone accordingly." if emotion else ""

    system_prompt = f"""You are a compassionate mental health support assistant.
    Use the following retrieved counseling examples as guidance for tone and content,
    but DO NOT copy them verbatim — synthesize your own empathetic, grounded response.
    If the user mentions self-harm or crisis, gently encourage them to reach out to a crisis helpline
    and to a trusted person, in addition to your response.
    {emotion_note}

    {context_block}
    """

    response = gpt_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content


In [12]:
print(generate_answer("I feel really overwhelmed with school and can't focus", emotion="anxiety"))

I’m really sorry you’re feeling so swamped right now. It can be incredibly hard to keep up with school when everything feels like it’s piling up, and the pressure to stay focused only adds to the stress. You’re not alone in this, and there are some practical steps you can try that often help to create a little breathing room in the midst of a busy schedule.

---

### 1. **Break it down into bite‑size pieces**
- **List everything** you need to do (assignments, readings, exams, projects). Seeing it on paper can make it feel less like a vague “mountain” and more like a series of small hills.
- **Prioritize**: Choose one or two items that are most urgent or carry the biggest weight for your grade. Anything that can wait, move it down the list.
- **Chunk it**: Take a larger task (e.g., “write paper”) and split it into concrete steps (research 2 sources, outline, write intro, etc.). Completing each micro‑step gives you a quick sense of progress.

### 2. **Use a timed‑focus method**
The **Pom